**Quick DAXM: depth of the Laue spots of a wire scan, without reconstruction of the images**

During a wire scan the intensity of a Laue spot drops to zero while a wire masks the ray going from the scattering grain to the pixel. This notebook:

1. finds the spots of the scan and collects their intensity profile over the images (in parallel);
2. finds the **sharp dips of the scattering signal**: the profile is the spot core minus the local background on the border of a small box, so the smooth dips of the fluorescence (same in the core and on the border) cancel;
3. converts the two edges of each dip into the position `ybeam` of the source along the beam, with the **wires trajectory parameters of the `.calib` file** (same geometrical model as the wires calibration and the reconstruction);
4. writes a `.cor` peak list with a depth column, and one `.cor` file per slice along the beam (ready for the indexation).

All parameters are in a YAML configuration file (template `config_template.yaml`). Core functions: `LaueTools/Daxm/quickdaxm.py`, workflow: `LaueTools/daxm_workflow.py`.

Units: `ybeam` in microns along the beam (> 0 deeper in the sample, 0 at the reference point of the wires calibration), `depth = ybeam * sin(sample tilt)` below the surface.

In [ ]:
# %matplotlib widget   # for jupyterlab interactive plots
import os
import numpy as np
import matplotlib.pyplot as plt

import LaueTools.daxm_workflow as DW
import LaueTools.Daxm.quickdaxm as QD

# 0 - Parameters

(cell tagged `parameters`)

In [ ]:
# configuration file of this scan (written at the first run, then yours: edit it and run again)
CONFIG = 'configs/<sample>_Zr20.yaml'

# scan: .scan file of the DAXM GUI or of the wires calibration notebook (images, detector, wire scan) ...
SCANFILE = '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0002/<sample>_Z1_GOI_Zr_20_register_0p8_calib.scan'
# ... wires trajectory parameters (.calib file of the wires calibration). None: dips only, no depth
CALIBFILE = '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0002/calibration/<sample>_Z1_GOI_Zr_20_register_0p8_test.calib'
# ... or, without .scan file (SCANFILE = None): BLISS hdf5 file of the dataset and scan number, e.g.
# HDF5_SCAN = ('/data/visitor/.../RAW_DATA/<sample>/<dataset>/<dataset>.h5', 1)
HDF5_SCAN = None
DETFILE = None        # None: detector calibration file of the .scan file (needed with CALIBFILE)
OUTPUT_FOLDER = None  # None: <PROCESSED_DATA mirror of the images folder>/daxm

# computation: 'kernel' (this notebook), 'local' (background job on this machine), 'magnifix' ... (SLURM)
RESOURCE = 'kernel'
NB_CPUS = None        # None: all available cpus
REDO = False          # True: compute again existing results

# 1 - Configuration file

Written from the template at the first run. Afterwards the file is kept with your edits; only the parameters given here are updated.

Without `.scan` file, give the BLISS hdf5 file of the dataset and the scan number instead (ascan or fscan of the wire motor; one image per file or all images in one file):
```python
cfg = DW.new_config(CONFIG, hdf5_scan=('/data/visitor/.../RAW_DATA/<sample>/<dataset>/<dataset>.h5', 1), detfile=..., calibfile=...)
```
Without `calibfile` the dips are found but not converted to depth (sections 2 to 4 only).

In [ ]:
changes = {'calibfile': CALIBFILE} if CALIBFILE else {}
if DETFILE:
    changes['detfile'] = DETFILE
if OUTPUT_FOLDER:
    changes['folder'] = OUTPUT_FOLDER
cfg = DW.new_config(CONFIG, scanfile=SCANFILE, hdf5_scan=None if SCANFILE else HDF5_SCAN, **changes)
DW.check_config(cfg)
print('results folder:', cfg['output']['folder'])
cfg['quickdaxm']

# 2 - Run

Results are saved in `<results folder>/quickdaxm.h5` and reloaded at the next run (`REDO = True` to compute again, e.g. after a change of the configuration file).

Same computation from a terminal: `python -m LaueTools.daxm_workflow quickdaxm <config.yaml> --resource magnifix`

In [ ]:
results = DW.run_quickdaxm(cfg, resource=RESOURCE, nb_cpus=NB_CPUS, redo=REDO)
spots, dips = results['spots'], results['dips']
print(f"{len(spots['X'])} spots, {int(np.sum(spots['nb_dips'] > 0))} with a position along the beam")

# 3 - Check the dips

Blue: spot minus local background (the profile that is analysed). Orange: local background, i.e. mostly fluorescence, with its smooth dips. Red and green lines: entry and exit edges of the dips found. Grey bands: images where each wire would mask a source at `ybeam = 0`; a dip shifted from its band comes from a deeper or shallower source.

If good dips are missed or bad ones are kept, change `low_fraction`, `min_edge_fraction`, `min_snr`, `boxsize` in the configuration file; the analysis of the collected profiles is redone in a second with the next cell but one (no need to read the images again).

In [ ]:
SPOTS = list(range(6))   # indices of spots (sorted by decreasing intensity)

fig, axes = plt.subplots(len(SPOTS) // 2, 2, figsize=(14, 3.2 * (len(SPOTS) // 2)))
for ax, spot in zip(axes.ravel(), SPOTS):
    QD.plot_profile(results, spot, ax)
fig.tight_layout()

In [ ]:
# OPTION: analyse again the collected profiles with the current configuration file (fast)
if 0:
    cfg = DW.load_config(CONFIG)
    results['params'] = {key: cfg['quickdaxm'][key] for key in QD.DEFAULT_PARAMS}
    results['dips'], results['spots'] = QD.analyse(results['signal'], results['xy'], QD.geometry_of_results(results), results['params'])
    QD.save_results(results, DW.output_files(cfg)['quickdaxm'])
    spots, dips = results['spots'], results['dips']

# 4 - Dips and wires calibration

Image index of the dips as a function of the pixel Y of the spots. Lines: shadow of each wire **calculated from the `.calib` file** for sources at `ybeam = 0` (solid) and 50 µm (dashed). Spots of a grain lie on a line parallel to them.

These lines replace the hand-tuned list `WP` of the former quick daxm notebooks: `wp_parameters()` gives the equivalent `[a, b]` of each wire (`image index = a / 450 * Y + b`, wires in the order of the `.calib` file) for comparison.

In [ ]:
QD.plot_abacus(results, ybeams=(0, 50))
geometry = QD.geometry_of_results(results)
if geometry is None:
    print('no wires calibration (CALIBFILE): dips only, the next sections need it')
else:
    print('equivalent WP parameters [a, b] of each wire:')
    for iw, ab in enumerate(geometry.wp_parameters()):
        print(f'  wire {iw}: [{ab[0]:.0f}, {ab[1]:.0f}]')

# 5 - Position of the spots along the beam

In [ ]:
if geometry is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.3, 1]})
    QD.plot_histogram(results, axes[0], step=2.)
    QD.plot_spots(results, axes[1])
    fig.tight_layout()

**Quality.** Each complete dip gives `ybeam` twice (entry and exit edges): `mismatch` is their difference. A spot masked by two wires during the scan gives two dips: `spread` is the difference of their `ybeam`. Both are a check of the wires calibration (a few microns expected). `extent` is the size of the source along the beam deduced from the blur of the edges.

In [ ]:
if geometry is not None:
    complete = np.isfinite(dips['mismatch'])
    twowires = spots['nb_dips'] > 1
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
    axes[0].hist(dips['mismatch'][complete], bins=np.arange(-10, 10.5, 0.5))
    axes[0].set_xlabel('mismatch entry - exit (µm)')
    axes[1].hist(spots['spread'][twowires], bins=np.arange(0, 20.5, 0.5))
    axes[1].set_xlabel('spread between 2 wires (µm)')
    axes[2].hist(spots['extent'][np.isfinite(spots['extent'])], bins=np.arange(0, 20.5, 0.5))
    axes[2].set_xlabel('extent of the source (µm, std)')
    fig.tight_layout()
    print(f"mismatch: median {np.median(dips['mismatch'][complete]):.1f} µm;  "
          f"spread between 2 wires: median {np.median(spots['spread'][twowires]):.1f} µm ({twowires.sum()} spots)")

# 6 - Peak lists

- `quickdaxm_spots.cor`: all spots with a position, with additional columns `ybeam`, `depth`, `extent` (µm), `wire`, `nb_dips`. 2theta and chi are computed for the source position of each spot.
- `depthslices/<name>_0000.cor ...`: one file per slice along the beam (`slice_step`, `slice_thickness`, `slice_min_spots` of the configuration file), 2theta and chi computed for the center of the slice; `depthslices/slices.yaml` lists the slices. The series can be indexed like a line scan with the indexation workflow (`notebooks/indexation`, `cor_folder` = `depthslices`).

In [ ]:
if geometry is not None:
    files = DW.write_quickdaxm_files(cfg, results)
    [(s['index'], s['ybeam'], s['nb_spots']) for s in files['slices'] if s['file']]

# 7 - Spots at a given position along the beam

Spots likely to belong to the grain(s) at `YBEAM` ± `TOLERANCE`, strongest first.

In [ ]:
if geometry is not None:
    YBEAM, TOLERANCE = 20., 3.   # microns

    indices = QD.select_spots(results, (YBEAM - TOLERANCE, YBEAM + TOLERANCE))
    print(f'{len(indices)} spots at ybeam = {YBEAM} +/- {TOLERANCE} µm (depth {geometry.depth(YBEAM):.1f} µm)')
    print('    X       Y      intensity  ybeam  wire')
    for k in indices[:20]:
        print(f"{spots['X'][k]:7.0f} {spots['Y'][k]:7.0f} {spots['intensity'][k]:10.0f} {spots['ybeam'][k]:6.1f} {int(spots['wire'][k]):4d}")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(spots['X'], spots['Y'], '.', color='0.8', ms=3)
    ax.scatter(spots['X'][indices], spots['Y'][indices], s=20, c='r')
    ax.set_xlim(0, 2 * geometry.ccd_params[1]); ax.set_ylim(2 * geometry.ccd_params[2], 0); ax.set_aspect('equal')
    ax.set_xlabel('X (pixel)'); ax.set_ylabel('Y (pixel)')
    # QD.write_cor(results, 'grain_ybeam20', cfg['output']['folder'], indices, ybeam_for_angles=YBEAM)   # .cor of these spots